# Split stratifié train/val/test - Cifer Fraud Detection

Exigence du sujet (points 12 et 18) : le split doit être stratifié pour
préserver le ratio de fraude dans chaque partition.
 
Répartition retenue : 70% train / 15% validation / 15% test
Stratification sur isFraud pour garantir la même proportion de fraude
dans les 3 ensembles.
 
Seed fixée (42) pour reproductibilité (exigence du sujet, point 18).
 
Auteur : Rasmané

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
 
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)
 
SEED = 42

## 0. CHARGEMENT DU DATASET AVEC FEATURES

In [ ]:
CHEMIN_ENTREE = r"C:\Users\hp\Documents\Fraude_detection\data\Cifer-echantillon-features.csv"
 
data1 = pd.read_csv(CHEMIN_ENTREE)
print(f"Dataset chargé : {data1.shape[0]:,} lignes, {data1.shape[1]} colonnes")
print(f"Taux de fraude global : {data1['isFraud'].mean()*100:.4f}%\n")

## 1. SÉPARATION FEATURES (X) / CIBLE (y)

In [ ]:
COLONNE_CIBLE = "isFraud"
 
X = data1.drop(columns=[COLONNE_CIBLE])
y = data1[COLONNE_CIBLE]
 
print(f"Nombre de features (X) : {X.shape[1]}")
print(f"Colonnes de X : {X.columns.tolist()}\n")

## 2. PREMIER SPLIT : TRAIN (70%) vs TEMP (30%)

In [ ]:
# stratify=y garantit que la proportion de fraude est identique dans
# train et temp (et donc, par transitivité, dans val et test ensuite)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=SEED
)

## 3. DEUXIÈME SPLIT : TEMP -> VALIDATION (15%) vs TEST (15%)

In [ ]:
# On coupe temp (30%) en deux moitiés égales (15% + 15% du total)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=SEED
)
 
del X_temp, y_temp

## 4. VÉRIFICATION DE LA STRATIFICATION

In [ ]:
print("=" * 70)
print("VÉRIFICATION DE LA STRATIFICATION")
print("=" * 70)
 
tableau_verif = pd.DataFrame({
    "ensemble": ["Train", "Validation", "Test"],
    "nb_lignes": [len(X_train), len(X_val), len(X_test)],
    "pct_du_total": [
        len(X_train) / len(data1) * 100,
        len(X_val) / len(data1) * 100,
        len(X_test) / len(data1) * 100,
    ],
    "nb_fraudes": [y_train.sum(), y_val.sum(), y_test.sum()],
    "taux_fraude_pct": [
        y_train.mean() * 100,
        y_val.mean() * 100,
        y_test.mean() * 100,
    ],
})
print(tableau_verif.round(4))
 
print(f"\nTaux de fraude dataset complet : {y.mean()*100:.4f}%")
print("-> Les taux ci-dessus doivent être quasi identiques à celui du")
print("   dataset complet (petites variations possibles dues aux arrondis).")
 
ecart_max = max(
    abs(y_train.mean() - y.mean()),
    abs(y_val.mean() - y.mean()),
    abs(y_test.mean() - y.mean())
)
print(f"\nÉcart maximal observé entre un ensemble et le taux global : "
      f"{ecart_max*100:.5f} points de %")
if ecart_max < 0.001:
    print("Stratification réussie : écarts négligeables.")
else:
    print("ATTENTION : écart plus important que prévu, à vérifier.")

## 6. SAUVEGARDE DES 6 FICHIERS (X_train, X_val, X_test, y_train, y_val, y_test)

In [ ]:
DOSSIER_SORTIE = r"C:\Users\hp\Documents\Fraude_detection\data"
 
X_train.to_csv(f"{DOSSIER_SORTIE}/X_train.csv", index=False)
X_val.to_csv(f"{DOSSIER_SORTIE}/X_val.csv", index=False)
X_test.to_csv(f"{DOSSIER_SORTIE}/X_test.csv", index=False)
y_train.to_csv(f"{DOSSIER_SORTIE}/y_train.csv", index=False)
y_val.to_csv(f"{DOSSIER_SORTIE}/y_val.csv", index=False)
y_test.to_csv(f"{DOSSIER_SORTIE}/y_test.csv", index=False)
 
print(f"\n{'='*70}")
print("FICHIERS SAUVEGARDÉS")
print(f"{'='*70}")
print(f"X_train.csv : {X_train.shape}")
print(f"X_val.csv   : {X_val.shape}")
print(f"X_test.csv  : {X_test.shape}")
print(f"y_train.csv : {y_train.shape}")
print(f"y_val.csv   : {y_val.shape}")
print(f"y_test.csv  : {y_test.shape}")
print(f"\nTous dans : {DOSSIER_SORTIE}")
 

print("""
NOTE MÉTHODOLOGIQUE (à reprendre dans le rapport) :
Le dataset a été divisé en trois ensembles (70% entraînement, 15%
validation, 15% test) via un split stratifié sur la variable cible
isFraud, garantissant une proportion de fraude identique dans chaque
partition. La graine aléatoire a été fixée (seed=42) pour assurer la
reproductibilité des résultats. L'ensemble de validation sera utilisé
pour comparer les stratégies de gestion du déséquilibre et pour la
recherche d'hyperparamètres ; l'ensemble de test, non utilisé avant
l'évaluation finale, sert à mesurer la performance de généralisation
du modèle retenu.
""")